# Beyond the Grinberg Lattice — Reproducibility Notebook

This notebook is a paper-aligned scaffold for reproducing the main delayed-lattice calculations reported in `beyond-grinberg-lattice.md`.

It is intentionally narrow. It does **not** validate paranormal or ontological claims. It reproduces the smaller scientific claim that a delayed lattice model can separate coherent switching, fragmentation, and decoder-lag effects under partial observation.

## 0. Setup
Run this notebook from the package directory or from a repository root that contains this package as a subdirectory.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
FINAL_PAPER = ROOT / 'final_paper'
if str(FINAL_PAPER) not in sys.path:
    sys.path.insert(0, str(FINAL_PAPER))

print('root:', ROOT)
print('final_paper exists:', FINAL_PAPER.exists())

## 1. Imports
Core delayed-lattice model and helper functions.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lattice_core import (
    LatticeConfig,
    run_sim,
    summarize,
    classify,
    evaluate_alias_decoder,
)

## 2. Baseline single-case reproduction
This reproduces the main coherent-switch example used in the manuscript.

In [ ]:
cfg = LatticeConfig()
result = run_sim(cfg)
summary = summarize(result)
summary

In [ ]:
ts = pd.DataFrame(result.rows)
ts.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(ts['t'], ts['m'], lw=2)
axes[0].axvline(summary['switch_step'], ls='--', color='crimson')
axes[0].set_ylabel('m(t)')
axes[0].set_title('Baseline coherent-switch trace')

axes[1].plot(ts['t'], ts['fragmentation'], color='darkorange', lw=2)
axes[1].axvline(summary['switch_step'], ls='--', color='crimson')
axes[1].set_ylabel('F(t)')

axes[2].plot(ts['t'], ts['matched_error'], label='matched', lw=2)
axes[2].plot(ts['t'], ts['lag_error'], label='lagged', lw=2)
axes[2].axvline(summary['switch_step'], ls='--', color='crimson')
axes[2].set_ylabel('abs error')
axes[2].set_xlabel('time step')
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()

## 3. Coarse phase grid
This section reproduces the qualitative phase split between fragmented partial forcing and coherent global forcing.

In [ ]:
J_VALUES = [0.20, 0.30, 0.45, 0.60]
PULSE_FRAC_VALUES = [0.45, 0.60, 0.80, 1.00]
SEEDS = [7, 11, 19]

rows = []
for J in J_VALUES:
    for pulse_fraction in PULSE_FRAC_VALUES:
        for seed in SEEDS:
            cfg = LatticeConfig(J=J, pulse_fraction=pulse_fraction, seed=seed)
            try:
                s = summarize(run_sim(cfg))
                phase = classify(s)
            except RuntimeError:
                phase = 'no_switch'
            rows.append({'J': J, 'pulse_fraction': pulse_fraction, 'seed': seed, 'phase': phase})

phase_df = pd.DataFrame(rows)
phase_df.head()

In [ ]:
majority = (phase_df.groupby(['J', 'pulse_fraction'])['phase']
            .agg(lambda s: s.value_counts().index[0])
            .unstack())
majority

## 4. Alias-decoder test
This section checks whether sector-conditioned decoding remains necessary when the sign of the boundary observable is removed.

In [ ]:
coherent_metrics = evaluate_alias_decoder(LatticeConfig(observed=16, seed=101))
fragmented_metrics = evaluate_alias_decoder(
    LatticeConfig(observed=16, J=0.20, noise=0.09, pulse_amp=-1.80, pulse_end=1700, pulse_fraction=0.45, seed=101)
)
pd.DataFrame([
    {'case': 'coherent_switch', **coherent_metrics},
    {'case': 'fragmented_switch', **fragmented_metrics},
])

## 5. Boundary-size sweep
This asks whether increasing boundary coverage rescues the lagged decoder.

In [ ]:
boundary_rows = []
for observed in [4, 8, 16, 24, 32]:
    metrics = evaluate_alias_decoder(LatticeConfig(observed=observed, seed=101))
    boundary_rows.append({
        'observed': observed,
        'transition_matched': metrics['transition_matched'],
        'transition_lagged': metrics['transition_lagged'],
        'transition_wrong': metrics['transition_wrong'],
        'transition_global': metrics['transition_global'],
    })
boundary_df = pd.DataFrame(boundary_rows)
boundary_df

In [ ]:
ax = boundary_df.plot(x='observed', y=['transition_matched', 'transition_lagged', 'transition_wrong'], marker='o', figsize=(8,5))
ax.set_ylabel('transition error')
ax.grid(alpha=0.25)
plt.show()

## 6. Multi-seed statistics
This is the notebook version of the seed-robustness summary.

In [ ]:
def mean_sd_ci(values):
    arr = np.asarray(values, dtype=float)
    mean = float(arr.mean())
    sd = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    ci95 = float(1.96 * sd / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
    return mean, sd, ci95

SEEDS = list(range(1, 31))
cases = {
    'coherent_switch': LatticeConfig(pulse_fraction=1.0),
    'fragmented_switch': LatticeConfig(J=0.20, noise=0.09, pulse_amp=-1.80, pulse_end=1700, pulse_fraction=0.45),
}

summary_rows = []
for case_name, base_cfg in cases.items():
    per_seed = []
    for seed in SEEDS:
        cfg = LatticeConfig(**{**base_cfg.__dict__, 'seed': seed})
        s = summarize(run_sim(cfg))
        per_seed.append(s)
    for key in ['transition_fragmentation', 'transition_matched_error', 'transition_lag_error', 'post_matched_error', 'mismatch_count']:
        vals = [row[key] for row in per_seed]
        mean, sd, ci95 = mean_sd_ci(vals)
        summary_rows.append({'case': case_name, 'metric': key, 'mean': mean, 'sd': sd, 'ci95_half_width': ci95})

seed_stats_df = pd.DataFrame(summary_rows)
seed_stats_df

## 7. Export helper
Optional: load the machine-readable outputs already generated by the standalone scripts.

In [ ]:
repro_root = PACKAGE_DIR / 'repro_outputs'
for path in sorted(repro_root.rglob('*')):
    if path.is_file():
        print(path.relative_to(ROOT))

## 8. Claim boundary reminder
Use this notebook to support the following restricted statement:

> A delayed lattice model can reproducibly distinguish coherent switching, fragmented transitions, and decoder-lag penalties under partial observation.

Do **not** use these outputs to claim literal holographic-brain implementation, confirmation of Grinberg's ontology, or validation of paranormal interpretations.